In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
import re
import sys
import types
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn.functional as F

sys.path.append("src")
from entropy_pruning import AttentionForecaster, UNILoRAClassifier, build_loaders, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
CFG = dict(
    data_dir="/raid/DATASETS/NCT-CRC-HE",
    img_size=224,
    batch_size=256,
    num_workers=4,
    seed=42,
)

dataset_name     = Path(CFG["data_dir"]).name
classifier_ckpt  = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{dataset_name}/uni_finetuned/best_model.pt")
forecaster_dir   = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{dataset_name}/forecaster")
multilayer_cache = Path(f"/raid/DATASETS/NCT-CRC-HE/data_cache/{dataset_name}_multilayer.h5")

# ── discover checkpoints ──────────────────────────────────────────────────────
ckpt_pattern = re.compile(r"forecaster_src(\d+)_tgt(\d+)\.pt")
checkpoints  = []
for p in sorted(forecaster_dir.glob("forecaster_src*_tgt*.pt")):
    m = ckpt_pattern.match(p.name)
    if m:
        checkpoints.append({"path": p, "src": int(m.group(1)), "tgt": int(m.group(2))})

src_layers = sorted(set(c["src"] for c in checkpoints))
tgt_layers = sorted(set(c["tgt"] for c in checkpoints))

print(f"Checkpoints found: {len(checkpoints)}")
for c in checkpoints:
    print(f"  src={c['src']:02d}  tgt={c['tgt']:02d}  →  {c['path'].name}")
print(f"\nSource layers to cache: {src_layers}")
print(f"Target layers to cache: {tgt_layers}")

In [ ]:
# ── build multi-layer cache (skip if already exists) ──────────────────────────
set_seed(CFG["seed"])

if multilayer_cache.exists():
    print(f"Cache already exists: {multilayer_cache}")
    with h5py.File(multilayer_cache, "r") as f:
        for split in f:
            print(f"  [{split}]", list(f[split].keys()))
else:
    print("Building multi-layer cache (this takes a few minutes)...")
    loaders = build_loaders(
        data_dir=CFG["data_dir"],
        img_size=CFG["img_size"],
        batch_size=CFG["batch_size"],
        num_workers=CFG["num_workers"],
        drop_last_train=False,
    )
    model = UNILoRAClassifier(loaders.n_classes).to(device)
    model.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    from entropy_pruning import build_attention_cache
    build_attention_cache(
        model=model,
        loaders={"train": loaders.train_loader, "val": loaders.val_loader, "test": loaders.test_loader},
        device=device,
        source_layers=src_layers,
        target_layers=tgt_layers,
        save_path=multilayer_cache,
    )
    del model
    torch.cuda.empty_cache()
    print(f"Cache saved → {multilayer_cache}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def load_test_tensors(h5_path, src_layer, tgt_layer):
    with h5py.File(h5_path, "r") as f:
        grp = f["test"]
        emb    = torch.from_numpy(grp[f"emb_layer{src_layer}"][:]).float()
        target = torch.from_numpy(grp[f"attn_layer{tgt_layer}"][:]).float()
    return TensorDataset(emb, target)

print("Test split sizes:")
ds_tmp = load_test_tensors(multilayer_cache, src_layers[0], tgt_layers[0])
print(f"  {len(ds_tmp):,} samples")

In [ ]:
def rankdata_2d(x: np.ndarray) -> np.ndarray:
    idx   = np.argsort(x, axis=1)
    ranks = np.empty_like(idx, dtype=float)
    np.put_along_axis(ranks, idx, np.arange(1, x.shape[1] + 1, dtype=float)[None], axis=1)
    return ranks

def spearman_rho(pred: np.ndarray, tgt: np.ndarray) -> float:
    rp, rt = rankdata_2d(pred), rankdata_2d(tgt)
    n      = pred.shape[1]
    rho    = 1 - 6 * ((rp - rt) ** 2).sum(axis=1) / (n * (n ** 2 - 1))
    return float(np.nanmean(rho))


@torch.no_grad()
def evaluate_checkpoint(ckpt_path, src_layer, tgt_layer, cache_path, batch_size=512):
    ds     = load_test_tensors(cache_path, src_layer, tgt_layer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

    forecaster = AttentionForecaster(embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1)
    forecaster.load_state_dict(torch.load(ckpt_path, map_location=device))
    forecaster.to(device).eval()

    preds, tgts = [], []
    kl_acc = 0.0
    for emb, tgt in loader:
        emb, tgt = emb.to(device), tgt.to(device)
        pred     = forecaster(emb)
        kl_acc  += F.kl_div((pred + 1e-8).log(), tgt + 1e-8, reduction="batchmean").item()
        preds.append(pred.cpu().numpy())
        tgts.append(tgt.cpu().numpy())

    pred_np = np.concatenate(preds)
    tgt_np  = np.concatenate(tgts)
    return {
        "rho": spearman_rho(pred_np, tgt_np),
        "kl":  kl_acc / len(loader),
    }

print("Eval function ready.")

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

records = []
for ckpt in tqdm(checkpoints, desc="evaluating"):
    metrics = evaluate_checkpoint(
        ckpt_path  = ckpt["path"],
        src_layer  = ckpt["src"],
        tgt_layer  = ckpt["tgt"],
        cache_path = multilayer_cache,
    )
    records.append({"src": ckpt["src"], "tgt": ckpt["tgt"], **metrics})
    print(f"  src={ckpt['src']:02d}  tgt={ckpt['tgt']:02d}  rho={metrics['rho']:.4f}  kl={metrics['kl']:.4f}")

df = pd.DataFrame(records)
Path("results").mkdir(exist_ok=True)
df.to_csv(f"results/src_tgt_ablation_{dataset_name}.csv", index=False)
df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

OURS_SRC, OURS_TGT = 2, 23
FONT = 13

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.size":       FONT,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── separate the two axes of variation ───────────────────────────────────────
# axis 1: fix src, vary tgt  (e.g. src=2)
# axis 2: fix tgt, vary src  (e.g. tgt=23)
df_tgt_var = df[df["src"] == OURS_SRC].sort_values("tgt")          # vary target
df_src_var = df[df["tgt"] == OURS_TGT].sort_values("src")          # vary source

has_both = len(df_tgt_var) > 1 and len(df_src_var) > 1
n_axes   = 2 if has_both else 1
figw     = 14 if has_both else 7

fig, axes = plt.subplots(1, n_axes, figsize=(figw, 5),
                         gridspec_kw={"wspace": 0.35})
if n_axes == 1:
    axes = [axes]

fig.suptitle(f"Source / Target Layer Ablation — {dataset_name}",
             fontsize=FONT + 2, fontweight="bold", y=1.02)

OURS_COLOR   = "#1f77b4"
OTHER_COLOR  = "#aec7e8"
OURS_EDGE    = "#0a3d6b"

def _bar_colors(srcs, tgts):
    return [
        OURS_COLOR if (s == OURS_SRC and t == OURS_TGT) else OTHER_COLOR
        for s, t in zip(srcs, tgts)
    ]

def _bar_edges(srcs, tgts):
    return [
        OURS_EDGE if (s == OURS_SRC and t == OURS_TGT) else "white"
        for s, t in zip(srcs, tgts)
    ]

def _bar_lws(srcs, tgts):
    return [
        2.0 if (s == OURS_SRC and t == OURS_TGT) else 0.5
        for s, t in zip(srcs, tgts)
    ]

# ── plot 1: vary target layer (fix src=OURS_SRC) ─────────────────────────────
if len(df_tgt_var) > 0:
    ax = axes[0]
    x  = np.arange(len(df_tgt_var))
    xlabels = [f"Layer {t}" for t in df_tgt_var["tgt"]]
    colors  = _bar_colors(df_tgt_var["src"], df_tgt_var["tgt"])
    edges   = _bar_edges(df_tgt_var["src"], df_tgt_var["tgt"])
    lws     = _bar_lws(df_tgt_var["src"], df_tgt_var["tgt"])

    bars = ax.bar(x, df_tgt_var["rho"], color=colors, edgecolor=edges,
                  linewidth=lws, width=0.6, zorder=3)

    ymin = df_tgt_var["rho"].min()
    for bar, val, ew, ec in zip(bars, df_tgt_var["rho"], lws, edges):
        fw = "bold" if ec == OURS_EDGE else "normal"
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=FONT - 2, fontweight=fw)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=FONT - 1)
    ax.set_ylabel("Spearman ρ  (test)", fontsize=FONT)
    ax.set_title(f"Varying target layer  (source = layer {OURS_SRC})",
                 fontsize=FONT)
    ax.set_ylim(ymin * 0.97, df_tgt_var["rho"].max() * 1.04)
    ax.grid(axis="y", alpha=0.25, zorder=0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

# ── plot 2: vary source layer (fix tgt=OURS_TGT) ─────────────────────────────
if len(df_src_var) > 0 and has_both:
    ax = axes[1]
    x  = np.arange(len(df_src_var))
    xlabels = [f"Layer {s}" for s in df_src_var["src"]]
    colors  = _bar_colors(df_src_var["src"], df_src_var["tgt"])
    edges   = _bar_edges(df_src_var["src"], df_src_var["tgt"])
    lws     = _bar_lws(df_src_var["src"], df_src_var["tgt"])

    bars = ax.bar(x, df_src_var["rho"], color=colors, edgecolor=edges,
                  linewidth=lws, width=0.6, zorder=3)

    ymin = df_src_var["rho"].min()
    for bar, val, ew, ec in zip(bars, df_src_var["rho"], lws, edges):
        fw = "bold" if ec == OURS_EDGE else "normal"
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=FONT - 2, fontweight=fw)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=FONT - 1)
    ax.set_ylabel("Spearman ρ  (test)", fontsize=FONT)
    ax.set_title(f"Varying source layer  (target = layer {OURS_TGT})",
                 fontsize=FONT)
    ax.set_ylim(ymin * 0.97, df_src_var["rho"].max() * 1.04)
    ax.grid(axis="y", alpha=0.25, zorder=0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

# ── legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=OURS_COLOR,  edgecolor=OURS_EDGE, linewidth=1.5,
                   label=f"Selected config  (src={OURS_SRC}, tgt={OURS_TGT})"),
    mpatches.Patch(facecolor=OTHER_COLOR, edgecolor="white",   linewidth=0.5,
                   label="Other configurations"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2,
           bbox_to_anchor=(0.5, -0.08), fontsize=FONT - 1, framealpha=0.9)

plt.tight_layout()
out_path = f"results/src_tgt_ablation_{dataset_name}.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

In [ ]:
print("Summary table:")
print(df.sort_values("rho", ascending=False).to_string(index=False))